# 🚗 AI/ML-Based Predictive Maintenance & Vehicle Health Monitoring

**Module 3 Connectivity Project — Project 1**

Goal: predict whether an engine requires maintenance from sensor readings, estimate failure probability, calculate a vehicle health score, and generate a maintenance alert.

This notebook uses a public Predictive Maintenance Engine Sensor Dataset from Hugging Face. The dataset contains 19,535 engine records and the target `Engine Condition` (0 = normal, 1 = maintenance/fault condition according to the dataset description).

**Dataset:** https://huggingface.co/datasets/mukherjee78/predictive-maintenance-engine-data

> Note: the project brief lists vibration, battery voltage, vehicle speed and operating hours as possible inputs. This dataset does not provide those columns, so this implementation uses the sensor columns actually present in the dataset rather than fabricating measurements.


In [ ]:
# Install required packages
!pip -q install xgboost shap


In [ ]:
# Imports
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from xgboost import XGBClassifier

RANDOM_STATE = 42


## 1. Load the dataset


In [ ]:
# Direct CSV download from Hugging Face
DATA_URL = 'https://huggingface.co/datasets/mukherjee78/predictive-maintenance-engine-data/resolve/main/raw_data.csv'

df = pd.read_csv(DATA_URL)

print('Dataset shape:', df.shape)
display(df.head())


In [ ]:
print('Columns:')
for col in df.columns:
    print('-', col)

print('\nMissing values:')
display(df.isnull().sum())

print('\nTarget distribution:')
display(df['Engine Condition'].value_counts().sort_index())


## 2. Basic exploratory data analysis


In [ ]:
display(df.describe().T)

df.hist(figsize=(14, 10), bins=30)
plt.suptitle('Sensor Feature Distributions', fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 7))
corr = df.corr(numeric_only=True)
plt.imshow(corr, aspect='auto')
plt.colorbar(label='Correlation')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=60, ha='right')
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()


## 3. Prepare features and target

The target is `Engine Condition`. We use the sensor columns as model inputs.


In [ ]:
TARGET = 'Engine Condition'

X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].astype(int).copy()

feature_names = X.columns.tolist()

print('Features:', feature_names)
print('Target:', TARGET)
print('Classes:', sorted(y.unique()))


## 4. Train / test split

Stratification keeps the class distribution approximately consistent between training and testing data.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print('Training samples:', len(X_train))
print('Testing samples :', len(X_test))


## 5. Build ML models


In [ ]:
models = {
    'Logistic Regression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),

    'Random Forest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_split=2,
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),

    'XGBoost': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric='logloss',
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
}


## 6. Train and evaluate all models


In [ ]:
results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    }

    results.append(metrics)
    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
    
display(results_df.style.format({c: '{:.4f}' for c in results_df.columns if c != 'Model'}))


## 7. Detailed evaluation of the Random Forest model


In [ ]:
rf_model = trained_models['Random Forest']
    
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, rf_pred, digits=4))

ConfusionMatrixDisplay.from_predictions(y_test, rf_pred)
plt.title('Random Forest — Confusion Matrix')
plt.show()

RocCurveDisplay.from_predictions(y_test, rf_prob)
plt.title('Random Forest — ROC Curve')
plt.show()


## 8. Feature importance — Explainable AI (XAI)


In [ ]:
rf_estimator = rf_model.named_steps['model']
importances = pd.Series(rf_estimator.feature_importances_, index=feature_names).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
importances.plot(kind='barh')
plt.xlabel('Importance')
plt.title('Random Forest Feature Importance')
plt.tight_layout()
plt.show()

display(importances.sort_values(ascending=False).to_frame('Importance'))


## 9. Failure probability → Vehicle Health Score

For this project demonstration:

- **Failure probability** = model probability of the maintenance/fault class.
- **Health score** = `100 × (1 − failure probability)`.
- Alert thresholds are demonstration thresholds and can be calibrated using validation data in a production system.


In [ ]:
def health_status(failure_probability):
    if failure_probability >= 0.70:
        return 'CRITICAL'
    elif failure_probability >= 0.40:
        return 'WARNING'
    else:
        return 'HEALTHY'


monitoring_output = X_test.copy().reset_index(drop=True)
monitoring_output['Actual Condition'] = y_test.reset_index(drop=True)
monitoring_output['Failure Probability'] = rf_prob
monitoring_output['Vehicle Health Score'] = (100 * (1 - rf_prob)).clip(0, 100)
monitoring_output['Status'] = monitoring_output['Failure Probability'].apply(health_status)
monitoring_output['Maintenance Alert'] = monitoring_output['Status'].apply(
    lambda x: 'INSPECTION REQUIRED' if x in ['WARNING', 'CRITICAL'] else 'NO IMMEDIATE ACTION'
)

display(monitoring_output.head(20))


## 10. Single-vehicle prediction function


In [ ]:
def predict_vehicle_health(sensor_values, model=rf_model):
    """
    sensor_values: dictionary containing the dataset's six sensor features.
    Returns failure probability, health score and maintenance status.
    """
    row = pd.DataFrame([sensor_values], columns=feature_names)
    failure_probability = float(model.predict_proba(row)[0, 1])
    predicted_condition = int(model.predict(row)[0])
    health_score = float(np.clip(100 * (1 - failure_probability), 0, 100))
    status = health_status(failure_probability)
    
    return {
        'Predicted Condition': predicted_condition,
        'Failure Probability (%)': round(failure_probability * 100, 2),
        'Vehicle Health Score (%)': round(health_score, 2),
        'Status': status,
        'Maintenance Alert': 'INSPECTION REQUIRED' if status != 'HEALTHY' else 'NO IMMEDIATE ACTION'
    }


## 11. Test with a real sample from the dataset


In [ ]:
sample_vehicle = X_test.iloc[0].to_dict()

print('Sensor values:')
for key, value in sample_vehicle.items():
    print(f'  {key}: {value}')

print('\nPrediction:')
prediction = predict_vehicle_health(sample_vehicle)
for key, value in prediction.items():
    print(f'  {key}: {value}')


## 12. Manual sensor-input prediction


In [ ]:
# Change these values to test another hypothetical vehicle.
# Keep the feature names exactly as they appear in the dataset.

manual_vehicle = {
    'Engine rpm': 900,
    'Lub oil pressure': 3.5,
    'Fuel pressure': 8.0,
    'Coolant pressure': 2.5,
    'lub oil temp': 80.0,
    'Coolant temp': 80.0
}

manual_prediction = predict_vehicle_health(manual_vehicle)

print('===== VEHICLE HEALTH MONITOR =====')
print(f"Vehicle Health: {manual_prediction['Vehicle Health Score (%)']}%")
print(f"Failure Probability: {manual_prediction['Failure Probability (%)']}%")
print(f"Status: {manual_prediction['Status']}")
print(f"Maintenance Alert: {manual_prediction['Maintenance Alert']}")


## 13. Generate maintenance-alert summary

This converts model output into a simple dashboard-style result similar to the project brief: health percentage, failure probability, status and contributing factors.


In [ ]:
def generate_alert(sensor_values, model=rf_model, top_n=3):
    row = pd.DataFrame([sensor_values], columns=feature_names)
    probability = float(model.predict_proba(row)[0, 1])
    score = float(np.clip(100 * (1 - probability), 0, 100))
    status = health_status(probability)

    # Global RF feature importance is used as a simple XAI contribution proxy.
    # For exact per-record explanations, use SHAP in the optional cell below.
    top_factors = importances.sort_values(ascending=False).head(top_n).index.tolist()

    return {
        'Vehicle Health': f'{score:.1f}%',
        'Failure Probability': f'{probability * 100:.1f}%',
        'Status': status,
        'Maintenance Alert': 'INSPECTION REQUIRED' if status != 'HEALTHY' else 'NO IMMEDIATE ACTION',
        'Major contributing factors': top_factors
    }

alert = generate_alert(manual_vehicle)

print('===== MAINTENANCE ALERT =====')
for key, value in alert.items():
    print(f'{key}: {value}')


## 14. Optional SHAP explanation (XAI)

Run this cell if you want a local explanation for one prediction. SHAP shows how each sensor pushes the prediction toward or away from the maintenance/fault class.


In [ ]:
import shap

X_train_imputed = pd.DataFrame(
    rf_model.named_steps['imputer'].transform(X_train),
    columns=feature_names
)

X_sample = pd.DataFrame([manual_vehicle], columns=feature_names)
X_sample_imputed = pd.DataFrame(
    rf_model.named_steps['imputer'].transform(X_sample),
    columns=feature_names
)

explainer = shap.TreeExplainer(rf_estimator)
shap_values = explainer.shap_values(X_sample_imputed)

if isinstance(shap_values, list):
    values_for_class_1 = shap_values[1][0]
else:
    values_for_class_1 = shap_values[0, :, 1] if shap_values.ndim == 3 else shap_values[0]

explanation = pd.DataFrame({
    'Feature': feature_names,
    'SHAP contribution': values_for_class_1
}).sort_values('SHAP contribution', key=np.abs, ascending=False)

display(explanation)


## 15. Save the trained model and monitoring results


In [ ]:
import joblib

joblib.dump(rf_model, 'predictive_maintenance_random_forest.pkl')
monitoring_output.to_csv('vehicle_health_predictions.csv', index=False)

print('Saved: predictive_maintenance_random_forest.pkl')
print('Saved: vehicle_health_predictions.csv')


## 16. Download outputs in Google Colab


In [ ]:
from google.colab import files

files.download('predictive_maintenance_random_forest.pkl')
# Run the next line separately if you also want the prediction CSV:
# files.download('vehicle_health_predictions.csv')


## Project architecture implemented

Vehicle/Engine Sensors → Data Preprocessing → ML Model → Failure Probability → Vehicle Health Score → Maintenance Alert

### Models included
- Logistic Regression
- Random Forest
- XGBoost

### XAI included
- Random Forest feature importance
- Optional SHAP local explanation

### Main outputs
- Failure probability
- Vehicle health score
- Status: HEALTHY / WARNING / CRITICAL
- Maintenance alert
- Major contributing sensor factors
